# Predicción de la Dirección de los Precios de Acciones con Redes Neuronales Profundas

In [1]:
# Importar librerías
import time
import numpy as np
import pandas as pd
import yfinance
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Descargar datos
df = yfinance.download("EURUSD=X", start="2010-01-01", end="2024-01-01", interval="1d")

# Si viene con MultiIndex, aplastar columnas
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Ahora las columnas deberían ser planas: Open, High, Low, Close, Adj Close, Volume
print(df.columns)

C:\Users\john_\AppData\Local\Temp\ipykernel_26516\309508850.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yfinance.download("SPY", start="2010-01-01", end="2024-01-01", interval="1d")
[*********************100%***********************]  1 of 1 completed

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')


In [3]:
# Graficar Datos
fig = make_subplots(rows=2, cols=1, row_heights=[0.80, 0.20])

fig.add_trace(go.Candlestick(x=df.index, open=df["Open"], high=df["High"], low=df["Low"], close=df["Close"], name="S&P 500"), 
              row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df["Volume"], name="Volumen"), row=2, col=1)

# Configurar diseño y mostrar la figura
fig.update_layout(height=600, width=1000, title="Gráfico de Velas y Volumen", xaxis_rangeslider_visible=False)
fig.show()

## Agregar Indicadores Técnicos

### Promedios Móviles

In [4]:
# Agregar diferentes Promedios Móviles
ventanas = [9, 10, 14, 20, 21, 50]
for i in ventanas:
    df[f"MA_{i}"] = df["Close"].rolling(i, min_periods=i).mean()
    
# Datos con nuevas variables
df

Price,Close,High,Low,Open,Volume,MA_9,MA_10,MA_14,MA_20,MA_21,MA_50
Date,,,,,,,,,,,
2010-01-04,85.279221,85.324368,83.909697,84.556835,118944600,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,85.504929,85.542555,84.917991,85.226513,111579900,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-06,85.565155,85.775850,85.354460,85.422181,116074400,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-07,85.926338,86.031686,85.166326,85.407121,131091100,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-08,86.212303,86.249930,85.527544,85.700613,126402800,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2023-12-22,463.589142,465.282405,461.680580,463.794673,67160400,460.199402,459.214557,455.416931,452.155630,451.759145,434.895889
2023-12-26,465.546692,466.456931,463.921948,464.000266,55387000,461.659281,460.734131,456.877648,453.281458,452.793300,435.794243
2023-12-27,466.388428,466.535236,464.802843,465.341149,68000300,462.519487,462.132196,458.526600,454.427441,453.905599,436.620919


In [5]:
# Gráficar 
fig = go.Figure()
for i in ventanas:
    fig.add_trace(go.Scatter(x=df.index, y=df[f"MA_{i}"], name=f"Promedio Móvil {i}"))
# Agregar el precio de cierre
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close", opacity=0.5))
fig.update_layout(title="Diferentes Promedios Móviles")
fig.show()

### Indicador RSI

In [6]:
# Implementar Indicador
def Relative_Strength_Index(df: pd.DataFrame, longitud: int = 14) -> pd.Series:

    """
    El Índice de Fuerza Relativa (RSI) es un indicador de impulso utilizado en el análisis 
    técnico que mide la magnitud de los cambios recientes en los precios para evaluar las 
    condiciones de sobrecompra o sobreventa en el precio de una acción u otro activo. 
    """

    # Calcular
    Delta = df["Close"].diff(periods = 1)
    Gain = Delta.where(Delta >= 0, 0)
    Loss = np.abs(Delta.where(Delta < 0, 0))
    avg_gain = Gain.ewm(alpha = 1 / longitud, min_periods=longitud).mean()
    avg_loss = Loss.ewm(alpha = 1 / longitud, min_periods=longitud).mean()
    RS = avg_gain/avg_loss
    RSI = pd.Series(np.where(RS == 0, 100, 100 - (100 / (1 + RS))), name = "RSI", index = df.index)

    return RSI

In [7]:
# Gráficar
ventanas = [9, 10, 14, 20, 21, 50]
for i in ventanas:
    df[f"RSI_{i}"] = Relative_Strength_Index(df, longitud=i)

fig = make_subplots(rows=3, cols=2)
fig.add_trace(go.Scatter(x=df.index, y=df["RSI_14"], name="RSI 9"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["RSI_14"], name="RSI 10"), row=1, col=2)
fig.add_trace(go.Scatter(x=df.index, y=df["RSI_14"], name="RSI 14"), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["RSI_14"], name="RSI 20"), row=2, col=2)
fig.add_trace(go.Scatter(x=df.index, y=df["RSI_14"], name="RSI 21"), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["RSI_14"], name="RSI 50"), row=3, col=2)

fig.update_layout(title="Análisis de RSI para diferentes ventanas de tiempo")
fig.show()

### Indicador MACD 

In [8]:
# Calcular Indicador
df["EMA_12"] = pd.Series(df["Close"].ewm(span=12, min_periods=12).mean())
df["EMA_26"] = pd.Series(df["Close"].ewm(span=26, min_periods=26).mean())
df["MACD"] = df["EMA_12"] - df["EMA_26"]
df["MACD_signal"] = df["MACD"].ewm(span=9, min_periods=9).mean()

# Realizar Gráfico
fig = make_subplots(rows=2, cols=1)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["EMA_12"], name="EMA 12"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["EMA_26"], name="EMA 26"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["MACD"], name="MACD"), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["MACD_signal"], name="Señal"), row=2, col=1)
fig.show()

## Limpiar Datos y Ajustarlos

In [9]:
# Ajustar Datos
df = df.shift(periods=1)
df["Prev_Close"] = df["Close"]
df["Dirección Close"] = (df["Close"] >= df["Close"].shift(periods=1)).astype(int)
# Eliminar valores faltantes
df.dropna(inplace=True)
df

Price,Close,High,Low,Open,Volume,MA_9,MA_10,MA_14,MA_20,MA_21,...,RSI_14,RSI_20,RSI_21,RSI_50,EMA_12,EMA_26,MACD,MACD_signal,Prev_Close,Dirección Close
Date,,,,,,,,,,,,,,,,,,,,,
2010-03-17,87.596848,87.679616,86.904556,87.145352,168673000.0,86.402075,86.212282,85.525368,84.854045,84.745652,...,75.054591,68.287589,67.526096,59.109101,85.973506,84.858361,1.115145,0.823763,87.596848,1
2010-03-18,88.116081,88.402030,87.604391,87.860239,177468100.0,86.774975,86.573476,85.870974,85.111395,85.009380,...,77.195295,70.133334,69.327942,60.327556,86.303199,85.104534,1.198665,0.898971,88.116081,1
2010-03-19,88.070915,88.243984,87.717246,88.123589,196509100.0,87.008248,86.904569,86.209594,85.342030,85.252324,...,76.579674,69.761556,68.978372,60.168418,86.575201,85.328357,1.246844,0.968714,88.070915,0
2010-03-22,87.625122,88.622493,87.285106,87.625122,226641100.0,87.190314,87.069936,86.454558,85.541724,85.450749,...,70.595483,66.119973,65.552836,58.611251,86.736750,85.501417,1.235334,1.022141,87.625122,0
2010-03-23,88.093575,88.252252,87.073537,87.126428,184477800.0,87.408542,87.280640,86.716322,85.764087,85.663241,...,72.984513,67.969442,67.342299,59.728801,86.945518,85.696486,1.249032,1.067590,88.093575,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-22,462.659363,462.933414,458.881338,461.318438,86667500.0,458.728492,457.716245,454.102308,451.167645,450.805209,...,70.642807,69.091218,68.755186,61.739218,456.453529,448.726777,7.726752,7.278257,462.659363,1
2023-12-26,463.589142,465.282405,461.680580,463.794673,67160400.0,460.199402,459.214557,455.416931,452.155630,451.759145,...,71.405182,69.657536,69.300348,62.015910,457.551316,449.827693,7.723622,7.367330,463.589142,1
2023-12-27,465.546692,466.456931,463.921948,464.000266,55387000.0,461.659281,460.734131,456.877648,453.281458,452.793300,...,72.995238,70.841539,70.440511,62.597018,458.781374,450.992063,7.789310,7.451726,465.546692,1


## Dividir Datos en Entrenamiento, Validación y Prueba

In [10]:
tamaño_prueba = 0.10

# Índices de división para el conjunto de datos
indice_division_prueba = int(df.shape[0] * (1 - tamaño_prueba))

# Conjuntos de datos de entrenamiento, validación y prueba
conjunto_entrenamiento = df.iloc[:indice_division_prueba].copy()
conjunto_prueba = df.iloc[indice_division_prueba+1:].copy()

# Gráficar
fig = go.Figure()
fig.add_trace(go.Scatter(x=conjunto_entrenamiento.index, y=conjunto_entrenamiento["Close"], name="Entrenamiento"))
fig.add_trace(go.Scatter(x=conjunto_prueba.index,  y=conjunto_prueba["Close"],  name="Prueba"))
fig.update_layout(title="División de Datos")
fig.show()

## Dividir en Características y Etiquetas

In [11]:
X_entrenamiento = conjunto_entrenamiento.drop(columns=["Close", "Dirección Close"])
y_entrenamiento = conjunto_entrenamiento["Dirección Close"]

X_prueba = conjunto_prueba.drop(columns=["Close", "Dirección Close"])
y_prueba =conjunto_prueba["Dirección Close"]

## Construir Red Neuronal

In [12]:
# Definir Semilla
torch.manual_seed(1)
np.random.seed(1)

# Escalar datos
scaler = StandardScaler()
X_entrenamiento = scaler.fit_transform(X_entrenamiento)
X_prueba = scaler.transform(X_prueba)

# Convertir datos a tensores
X_entrenamiento_tensor = torch.tensor(X_entrenamiento, dtype=torch.float32)
y_entrenamiento_tensor = torch.tensor(y_entrenamiento.values, dtype=torch.float32).reshape(-1, 1)

X_prueba_tensor = torch.tensor(X_prueba, dtype=torch.float32)
y_prueba_tensor = torch.tensor(y_prueba.values, dtype=torch.float32).reshape(-1, 1)

In [13]:
# Definir arquitectura de Red Neuronal Profunda
modelo = nn.Sequential(
    nn.Linear(in_features=X_entrenamiento_tensor.shape[1], out_features=X_entrenamiento_tensor.shape[1] * 4),
    nn.ReLU(),
    nn.Linear(in_features=X_entrenamiento_tensor.shape[1] * 4, out_features=X_entrenamiento_tensor.shape[1] * 4),
    nn.ReLU(),
    nn.Linear(in_features=X_entrenamiento_tensor.shape[1] * 4, out_features=X_entrenamiento_tensor.shape[1] * 4),
    nn.ReLU(),
    nn.Linear(in_features=X_entrenamiento_tensor.shape[1] * 4, out_features=1),
    nn.Sigmoid()
)
modelo

Sequential(
  (0): Linear(in_features=21, out_features=84, bias=True)
  (1): ReLU()
  (2): Linear(in_features=84, out_features=84, bias=True)
  (3): ReLU()
  (4): Linear(in_features=84, out_features=84, bias=True)
  (5): ReLU()
  (6): Linear(in_features=84, out_features=1, bias=True)
  (7): Sigmoid()
)

In [14]:
# Probar que funciona
modelo(X_entrenamiento_tensor)

tensor([[0.5204],
        [0.5228],
        [0.5229],
        ...,
        [0.5277],
        [0.5277],
        [0.5295]], grad_fn=<SigmoidBackward0>)

In [15]:
# Definir función de pérdida y optimizador
lossfunc = nn.BCELoss()
optimizer = optim.Adam(modelo.parameters(), lr=0.001)

# Guardar valores de pérdida y precisión
loss_values = []
accuracy_values = []

# Entrenamiento del modelo
num_epochs = 3000
for epoch in range(num_epochs):
    # Predecir y obtener error
    outputs = modelo(X_entrenamiento_tensor)
    loss = lossfunc(outputs, y_entrenamiento_tensor)
    
    # Paso backward y optimización
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Calcular y almacenar la pérdida
    loss_values.append(loss.item())
    
    # Calcular la precisión
    predicted = (outputs >= 0.5).float()
    accuracy = (predicted == y_entrenamiento_tensor).float().mean().item()
    accuracy_values.append(accuracy)
    
    # Imprimir pérdida y precisión cada 100 épocas
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')

Epoch [100/3000], Loss: 0.5001, Accuracy: 0.7596
Epoch [200/3000], Loss: 0.4357, Accuracy: 0.7951
Epoch [300/3000], Loss: 0.4072, Accuracy: 0.8111
Epoch [400/3000], Loss: 0.3843, Accuracy: 0.8268
Epoch [500/3000], Loss: 0.3627, Accuracy: 0.8374
Epoch [600/3000], Loss: 0.3482, Accuracy: 0.8476
Epoch [700/3000], Loss: 0.3477, Accuracy: 0.8480
Epoch [800/3000], Loss: 0.3158, Accuracy: 0.8556
Epoch [900/3000], Loss: 0.3083, Accuracy: 0.8617
Epoch [1000/3000], Loss: 0.2966, Accuracy: 0.8723
Epoch [1100/3000], Loss: 0.2901, Accuracy: 0.8707
Epoch [1200/3000], Loss: 0.2804, Accuracy: 0.8736
Epoch [1300/3000], Loss: 0.2716, Accuracy: 0.8787
Epoch [1400/3000], Loss: 0.2652, Accuracy: 0.8835
Epoch [1500/3000], Loss: 0.2709, Accuracy: 0.8819
Epoch [1600/3000], Loss: 0.2581, Accuracy: 0.8912
Epoch [1700/3000], Loss: 0.2584, Accuracy: 0.8870
Epoch [1800/3000], Loss: 0.2429, Accuracy: 0.8950
Epoch [1900/3000], Loss: 0.2441, Accuracy: 0.8979
Epoch [2000/3000], Loss: 0.2333, Accuracy: 0.9027
Epoch [21

In [16]:
# Predicciones
y_entrenamiento_pred = (modelo(X_entrenamiento_tensor).detach().numpy() >= 0.5).astype(int).flatten()
y_prueba_pred = (modelo(X_prueba_tensor).detach().numpy() >= 0.5).astype(int).flatten()

In [17]:
# Calcular métricas de rendimiento en el conjunto de prueba
accuracy_entrenamiento = accuracy_score(y_entrenamiento, y_entrenamiento_pred)
accuracy_prueba = accuracy_score(y_prueba, y_prueba_pred)

print(f'Accuracy en conjunto de entrenamiento: {accuracy_entrenamiento:.4f}')
print(f'Accuracy en conjunto de prueba: {accuracy_prueba:.4f}')

Accuracy en conjunto de entrenamiento: 0.9139
Accuracy en conjunto de prueba: 0.8069


### Recordatorio:

    - Las redes neuronales profundas son herramientas poderosas para anticipar la dirección del mercado al identificar patrones sutiles.
    - Se pueden tomar diferentes modelos para mejorar la precisión en la toma de decisiones a la hora de invertir.